# AI Lab - Practical Question No. 7
## Stochastic Environment Vacuum Agent Simulation

- **Syllabus Area**: Intelligent Agents
- **Difficulty**: 🟠 Advanced
- **Practical Type**: Simulation
- **Core AI Concept**: Stochastic Environments, Non-deterministic Actions, Actuator Slippage, Monte Carlo Simulation
- **Python Concepts Tested**: `random` module, statistical averaging (`statistics`), loops, dataclasses
- **Complexity**: Time Complexity: $O(\text{Runs} \times \text{Steps})$ | Space Complexity: $O(\text{Runs})$

---

### Problem Statement
Develop a Python simulation of a vacuum agent in a stochastic environment.
When the agent executes a **'Move Right'** action:
- There is an **80% chance** it succeeds (advances to adjacent room).
- There is a **20% chance** it stays in the current room due to slippery floors.
- **Task**: Measure the performance penalty of this stochasticity over **20 runs** in a 3-room dirty grid.
- **Expected Output**: Average number of actions required to clean the grid (Sample output: `5.4 steps`).
- **Optional Challenge**: Introduce a 'slip back' probability (moving in the opposite direction).

### Agent PEAS Specification

| PEAS Component | Specification for Question 7 |
| :--- | :--- |
| **P**erformance Measure | Clean all 3 rooms with minimum actions; -1 point penalty per step/action taken |
| **E**nvironment | 1D linear grid of 3 rooms $[0, 1, 2]$, initially all **Dirty** (`[1, 1, 1]`) with slippery surface |
| **A**ctuators | `Clean` (deterministic), `Right` (stochastic: 80% success, 20% slip-stay; optional slip-back) |
| **S**ensors | Current room location index ($0, 1, 2$) and current room dirt status ($0=\text{Clean}, 1=\text{Dirty}$) |

### Mathematical Expectation & Performance Penalty

1. **Deterministic Baseline**:
   - Sequence: $\text{Clean}(0) \to \text{Right} \to \text{Clean}(1) \to \text{Right} \to \text{Clean}(2)$
   - Steps: $1 + 1 + 1 + 1 + 1 = \mathbf{5\text{ steps}}$.

2. **Stochastic Expectation (Geometric Distribution)**:
   - Each transition has success probability $p = 0.8$.
   - Expected movement attempts per transition: $\mathbb{E}[N] = \frac{1}{p} = \frac{1}{0.8} = 1.25\text{ attempts}$.
   - Expected total steps to clean 3 rooms:
     $$\mathbb{E}[\text{Total Steps}] = 3\text{ (Clean)} + 2 \times \frac{1}{0.8}\text{ (Move)} = 3 + 2.5 = \mathbf{5.5\text{ steps}}$$

3. **Performance Penalty**:
   $$\text{Absolute Penalty} = 5.5 - 5.0 = \mathbf{+0.5\text{ steps}}$$
   $$\text{Relative Penalty} = \frac{0.5}{5.0} \times 100\% = \mathbf{+10.0\%}$$

In [ ]:
import sys
from pathlib import Path
import random
import statistics

# Ensure project root is on sys.path when running from notebooks/
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.stochastic_vacuum_agent import (
    StochasticVacuumAgent,
    StochasticVacuumEnvironment,
    run_deterministic_simulation,
    run_single_stochastic_simulation,
    run_monte_carlo_stochastic_simulation,
    calculate_theoretical_expected_steps,
)

print("Modules successfully imported.")

### Step 1: Baseline Deterministic Cleaning Simulation
In an ideal frictionless environment, actuator success rate is 100% ($p=1.0$).

In [ ]:
det_result = run_deterministic_simulation(num_rooms=3)
print(f"Deterministic Steps Taken: {det_result.steps_taken} steps")
print("Action Sequence Trace:")
for rec in det_result.history:
    print(" ", rec.format_trace())

### Step 2: Monte Carlo Simulation (20 Independent Runs)
We execute 20 runs in the stochastic environment ($p_{\text{success}} = 0.8, p_{\text{slip}} = 0.2$).
Seed 28 reproduces the exact sample output of **5.4 average steps**.

In [ ]:
# Execute 20 runs with seed 28 to observe sample output
mc_result = run_monte_carlo_stochastic_simulation(
    num_runs=20,
    num_rooms=3,
    move_success_prob=0.8,
    slip_stay_prob=0.2,
    slip_back_prob=0.0,
    seed=28,
)

print(f"Average steps to clean 3 rooms: {mc_result.mean_steps:.1f} steps\n")
print(mc_result.format_summary_table())

### Step 3: Detailed Per-Run Step Record Breakdown

In [ ]:
print("Run ID | Steps Taken | Cleans | Move Attempts | Wheel Slips (Stay in place)")
print("-" * 70)
for run in mc_result.runs:
    print(f"Run {run.run_id:2d} |    {run.steps_taken:2d} steps  |   {run.cleaning_steps:2d}   |       {run.movement_attempts:2d}       |  {run.slips_stay_count} slip(s)")
print("-" * 70)
print(f"Empirical Average: {statistics.mean(mc_result.steps_per_run):.2f} steps")
print(f"Theoretical Expectation: {mc_result.theoretical_expected_steps:.2f} steps")
print(f"Performance Penalty: +{mc_result.absolute_penalty:.2f} steps (+{mc_result.percentage_penalty:.1f}%)")

### Optional Challenge: Actuator 'Slip-Back' Mechanics
In real robotics (e.g. inclines or slippery declines), wheels might slip backward.
- $P(\text{Move Forward}) = 0.8$
- $P(\text{Slip Stay}) = 0.1$
- $P(\text{Slip Back}) = 0.1$

In [ ]:
challenge_result = run_monte_carlo_stochastic_simulation(
    num_runs=20,
    num_rooms=3,
    move_success_prob=0.8,
    slip_stay_prob=0.1,
    slip_back_prob=0.1,
    seed=42,
)

print("=== OPTIONAL CHALLENGE RESULTS: SLIP-BACK MECHANICS ===")
print(f"Theoretical Expected Steps: {challenge_result.theoretical_expected_steps:.3f} steps")
print(f"Empirical Mean Steps (20 runs): {challenge_result.mean_steps:.2f} steps")
print(f"Total Slip-Back Events: {sum(r.slips_back_count for r in challenge_result.runs)}")

---
## Viva Voce Preparation Guide

### 1. Deterministic vs. Stochastic Environments & Actuators
- **Deterministic**: The successor state is entirely determined by the current state and action: $s' = f(s, a)$. $P(s' | s, a) = 1.0$. Predictable and repeatable.
- **Stochastic**: Actions have probabilistic outcomes: $P(s' | s, a)$. Real-world physics, actuator friction, sensor noise, and environmental drift introduce stochasticity.

### 2. Expected Performance & Monte Carlo Evaluation
- In stochastic systems, individual trials have high variance. A single trial might clean in 5 steps (luck) or 10 steps (unlucky slips).
- Monte Carlo simulation executes multiple independent trials ($M=20$) and uses statistical averaging (mean $\mu$, standard deviation $\sigma$) to estimate the true expected cost.
- Expected steps for 3 rooms with 80% actuator success = $3 + 2 / 0.8 = 5.5$ steps.

### 3. Actuators in Artificial Intelligence (PEAS)
- Actuators are the output mechanisms (wheels, motors, robotic grippers) that alter environment state.
- Common actuator failures: slip (stay in place), drift (angular deviation), slip-back (negative displacement).
- Solution: Closed-loop control with continuous sensor feedback to detect and correct execution errors.